In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rahmasleam/bird-speciees-dataset")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/rahmasleam/bird-speciees-dataset/versions/1


In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt


os.listdir(path)

['Bird Speciees Dataset']

In [ ]:
root = path + "/Bird Speciees Dataset"
os.listdir(root)

['AMERICAN GOLDFINCH',
 'CARMINE BEE-EATER',
 'DOWNY WOODPECKER',
 'BARN OWL',
 'EMPEROR PENGUIN',
 'FLAMINGO']

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
BUFFER_SIZE = 1000
AUTOTUNE = tf.data.AUTOTUNE
SEED = 42
EPOCHS = 75
LR = 0.0001

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(root,
                                                  image_size=(IMG_SIZE, IMG_SIZE),
                                                  batch_size=BATCH_SIZE,
                                                  seed=SEED,
                                                  subset='training',
                                                  validation_split=0.2,
                                                  shuffle=True)

val_ds = tf.keras.utils.image_dataset_from_directory(root,
                                                  image_size=(IMG_SIZE, IMG_SIZE),
                                                  batch_size=BATCH_SIZE,
                                                  seed=SEED,
                                                  subset='validation',
                                                  validation_split=0.2,
                                                  shuffle=False)

Found 811 files belonging to 6 classes.
Using 649 files for training.
Found 811 files belonging to 6 classes.
Using 162 files for validation.


In [ ]:
class_names = train_ds.class_names
print(class_names)

['AMERICAN GOLDFINCH', 'BARN OWL', 'CARMINE BEE-EATER', 'DOWNY WOODPECKER', 'EMPEROR PENGUIN', 'FLAMINGO']


In [ ]:
plt.figure(figsize=(15, 12))
for images, labels in train_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
train_ds = train_ds.cache().shuffle(BUFFER_SIZE).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
def make_model(input_shape, num_classes, dropout_rate=0.2):
    inputs = keras.Input(shape=input_shape)

    # Bloco inicial com Conv2D e BatchNorm
    x = layers.Conv2D(32, 3, strides=2, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(64, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    previous_block_activation = x  # Salvando ativação anterior para adicionar ao bloco residual

    # Blocos intermediários com SeparableConv2D, BatchNorm e Resíduo
    for size in [128, 256, 512, 728]:
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        # MaxPooling reduzindo a resolução
        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

        # Conexão residual
        residual = layers.Conv2D(size, 1, strides=2, padding="same")(previous_block_activation)
        x = layers.add([x, residual])
        previous_block_activation = x  # Atualizando a ativação anterior para o próximo bloco

    # Último bloco convolucional
    x = layers.SeparableConv2D(1024, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # Pooling global para redução final
    x = layers.GlobalAveragePooling2D()(x)

    if num_classes == 2:
        activation = "sigmoid"
        units = 1
    else:
        activation = "softmax"
        units = num_classes

    x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(units, activation=activation)(x)

    return keras.Model(inputs, outputs)

In [ ]:
model = make_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=len(class_names))
tf.keras.utils.plot_model(model, show_shapes=True)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(train_ds, batch_size=BATCH_SIZE,
          epochs=EPOCHS, validation_data=val_ds)